# The best plain LoRA we can find: hyperparameter search before JEPA

Setting: **Gemma 3 1B IT + LoRA**, the same **50 training pairs** (5 per move, the same seed as notebook 13), the prompt `Rewrite in the Titan's voice: <x>`. There is no JEPA here. The aim is the strongest plain-LoRA recipe, so that a later JEPA comparison cannot be dismissed as "LoRA was not tuned".

| Stage | Configurations | Grid |
|---|---|---|
| **1** | 18 | learning rate {5e-5, 1e-4, 2e-4} × rank {8, 16, 32} (α = 2 × rank) × dropout {0.05, 0.15} |
| **2** | 4 | the best stage-1 setting × NEFTune noise {off, α = 5} × weight decay {0, 0.05} |
| **top 3** | 3 | the three lowest validation losses overall: translate the 100 **validation** sentences, run the fact checks, save the outputs for the blind judgement |

- **Each configuration:** up to 30 epochs, early stopping when validation loss has not improved for 3 epochs, best epoch kept.
- **Selection uses validation only;** the test set is untouched until the final comparison.
- **Checkpoints:** the summary (every configuration's per-epoch validation loss) is saved and mirrored after each configuration. A restarted job skips the finished ones, and the top 3 are retrained if their weights are not in memory.

In [ ]:
# Cell 0 — Hugging Face token: HF_TOKEN in the environment (Modal) or a Kaggle secret; never printed

import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from kaggle_secrets import UserSecretsClient
    for label in ("HF_TOKEN", "hf_token"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(label)
            break
        except Exception:
            pass
assert HF_TOKEN, "set HF_TOKEN (Modal secret) or add it under Kaggle Add-ons → Secrets"

## Cell 1 — Setup, grids, Hub mirror

In [ ]:
# Cell 1 — Packages, settings, grids, Hub mirror

!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import gc, json, random, re, shutil, statistics, time, urllib.request
from collections import Counter

import peft
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, login

DECODER = os.environ.get("DECODER", "google/gemma-3-1b-it")
DATA = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/titan_v1/"
N_PER_MOVE = int(os.environ.get("N_PER_MOVE", 5))
MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 30, 3, 4, 2
MAX_NEW_TOKENS, MAX_TARGET_TOKENS = 128, 128
INSTRUCTION = "Rewrite in the Titan's voice: "
SEED, TOP_K = 42, 3
GRID1 = [{"lr": lr, "r": r, "dropout": d, "neftune": 0.0, "wd": 0.0}
         for lr in (5e-5, 1e-4, 2e-4) for r in (8, 16, 32) for d in (0.05, 0.15)]
HUB_REPO, RUN_NAME = "skamalj/titan-stage1", "lora-sweep-v1"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DEVICE = torch.device("cuda:0")
MODEL_TAG = DECODER.split("/")[-1].replace("gemma-3-", "").replace("-it", "")
MULTIMODAL = MODEL_TAG not in ("1b", "270m")

login(HF_TOKEN)
api = HfApi(token=HF_TOKEN)
REV = api.model_info(DECODER, token=HF_TOKEN).sha
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), (transformers.__version__, peft.__version__)
cfg_name = lambda c: f"lr{c['lr']:g}_r{c['r']}_d{c['dropout']:g}_nef{c['neftune']:g}_wd{c['wd']:g}"


def hub_path(local):
    return f"runs/{RUN_NAME}/{os.path.relpath(local, WORK)}"


def hub_put(local):
    try:
        api.upload_file(repo_id=HUB_REPO, path_or_fileobj=local, path_in_repo=hub_path(local),
                        commit_message=f"{RUN_NAME}: {hub_path(local)}")
    except Exception as e:
        print(f"  (Hub mirror failed for {hub_path(local)}: {type(e).__name__}; the local copy is kept)")


def hub_get(local):
    if os.path.exists(local):
        return True
    try:
        if hub_path(local) not in api.list_repo_files(HUB_REPO):
            return False
        shutil.copy(hf_hub_download(HUB_REPO, hub_path(local), token=HF_TOKEN), local)
        return True
    except Exception as e:
        print(f"  (Hub restore failed: {type(e).__name__})")
        return False


def save_json(obj, path):
    json.dump(obj, open(path, "w"), ensure_ascii=False)
    hub_put(path)


print({"decoder": DECODER, "n_train": 10 * N_PER_MOVE, "stage1_configs": len(GRID1), "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0), "work": WORK})

## Cell 2 — Data (the same 50-pair subset as notebook 13)

In [ ]:
# Cell 2 — Data

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


FULL_TRAIN, VAL, TEST = fetch_jsonl("train.jsonl"), fetch_jsonl("validation.jsonl"), fetch_jsonl("test.jsonl")
rng = random.Random(SEED)
by_move = {}
for r in FULL_TRAIN:
    by_move.setdefault(r["move"], []).append(r)
TRAIN = []
for m in sorted(by_move):
    TRAIN += rng.sample(by_move[m], N_PER_MOVE)
TRAIN.sort(key=lambda r: r["id"])
print({"train": len(TRAIN), "validation": len(VAL)}, "| first ids:", [r["id"] for r in TRAIN[:5]])

## Cell 3 — Model builder, losses, NEFTune, generation, scores

- **Model:** each configuration loads a fresh base model (bf16) and a fresh LoRA, seeded identically.
- **NEFTune:** during training only, uniform noise is added to the input embeddings at scale α / √(L · d), where L is the sequence length and d the width.

In [ ]:
# Cell 3 — Model builder, losses, NEFTune, generation, scores

from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Gemma3ForConditionalGeneration

tok = AutoTokenizer.from_pretrained(DECODER, revision=REV, token=HF_TOKEN)
LOADER = Gemma3ForConditionalGeneration if MULTIMODAL else AutoModelForCausalLM
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EOS = tok.convert_tokens_to_ids("<end_of_turn>")
NEF = {"alpha": 0.0}
model = None


def build(cfg):
    global model
    if model is not None:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    base = LOADER.from_pretrained(DECODER, revision=REV, token=HF_TOKEN, dtype=torch.bfloat16, device_map={"": 0},
                                  attn_implementation="eager")
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    torch.manual_seed(SEED)
    model = get_peft_model(base, LoraConfig(r=cfg["r"], lora_alpha=2 * cfg["r"], lora_dropout=cfg["dropout"],
                                            task_type="CAUSAL_LM",
                                            target_modules=rf"(.*language_model\.layers|model\.layers)\..*\.({'|'.join(PROJ)})"))
    NEF["alpha"] = cfg["neftune"]
    model.get_input_embeddings().register_forward_hook(neftune_hook)
    return [p for n, p in model.named_parameters() if "lora_" in n]


def neftune_hook(module, args, output):
    if NEF["alpha"] and module.training:
        L, d = output.shape[1], output.shape[2]
        output = output + torch.empty_like(output).uniform_(-1, 1) * (NEF["alpha"] / (L * d) ** 0.5)
    return output


def prompt_ids(x):
    text = tok.apply_chat_template([{"role": "user", "content": INSTRUCTION + x}], tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False)["input_ids"]


def ce_loss(records):
    seqs, labels = [], []
    for r in records:
        p = prompt_ids(r["source"])
        t = tok(r["target"], add_special_tokens=False)["input_ids"][:MAX_TARGET_TOKENS] + [EOS]
        seqs.append(p + t)
        labels.append([-100] * len(p) + t)
    T = max(len(s) for s in seqs)
    ids = torch.tensor([s + [tok.pad_token_id] * (T - len(s)) for s in seqs], device=DEVICE)
    mask = torch.tensor([[1] * len(s) + [0] * (T - len(s)) for s in seqs], device=DEVICE)
    lab = torch.tensor([l + [-100] * (T - len(l)) for l in labels], device=DEVICE)
    extra = {"token_type_ids": torch.zeros_like(mask)} if MULTIMODAL else {}
    return model(input_ids=ids, attention_mask=mask, labels=lab, **extra).loss


@torch.no_grad()
def val_ce(records, batch=8):
    model.eval()
    return round(sum(ce_loss(records[i:i + batch]).item() * len(records[i:i + batch])
                     for i in range(0, len(records), batch)) / len(records), 4)


@torch.no_grad()
def translate(records, batch=16):
    model.eval()
    eos = model.generation_config.eos_token_id
    stops = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
    outs = []
    for i in range(0, len(records), batch):
        seqs = [prompt_ids(r["source"]) for r in records[i:i + batch]]
        T = max(len(s) for s in seqs)
        ids = torch.tensor([[tok.pad_token_id] * (T - len(s)) + s for s in seqs], device=DEVICE)
        mask = torch.tensor([[0] * (T - len(s)) + [1] * len(s) for s in seqs], device=DEVICE)
        gen = model.generate(input_ids=ids, attention_mask=mask, do_sample=False, num_beams=1,
                             max_new_tokens=MAX_NEW_TOKENS, eos_token_id=stops, pad_token_id=tok.pad_token_id,
                             temperature=None, top_p=None, top_k=None)
        outs += [tok.decode(row, skip_special_tokens=True).strip() for row in gen[:, T:].tolist()]
    return outs


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


# --- faithfulness checks: the fact checks of the dataset's own validator (raw/validate_titan.py), per row ---
ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
        "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
TENS = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
ORDINAL = {1: "first", 2: "second", 3: "third", 5: "fifth", 8: "eighth", 9: "ninth", 12: "twelfth"}
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]


def cardinal(n):
    if n < 20:
        return ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return TENS[t] + ("" if o == 0 else "-" + ONES[o])
    if n < 1000:
        h, r = divmod(n, 100)
        return ONES[h] + " hundred" + ("" if r == 0 else " " + cardinal(r))
    if n < 1000000:
        th, r = divmod(n, 1000)
        return cardinal(th) + " thousand" + ("" if r == 0 else " " + cardinal(r))
    return str(n)


def ordinal(n):
    if n in ORDINAL:
        return ORDINAL[n]
    if n < 20:
        return cardinal(n) + "th"
    if n >= 100:
        head, tail = n - n % 100, n % 100
        return cardinal(head) + ("th" if tail == 0 else " " + ordinal(tail))
    if n % 10 == 0:
        return cardinal(n)[:-1] + "ieth"
    t, o = divmod(n, 10)
    return TENS[t] + "-" + ordinal(o)


def num_words(n):
    c, o = cardinal(n), ordinal(n)
    forms = {c, c.replace("-", " "), c.replace(" hundred ", " hundred and "), o, o.replace("-", " ")}
    if 1000 <= n < 10000 and n % 100 == 0:
        forms.add(cardinal(n // 100) + " hundred")
    forms |= {15: {"quarter past", "a quarter"}, 30: {"half past", "half an hour", "half"},
              45: {"quarter to"}}.get(n, set())
    return forms


has_any = lambda low, forms: any(f in low for f in forms)


def time_ok(h, mm, low):
    H, M = int(h), int(mm)
    nxt = H % 12 + 1
    return (f"{H}:{mm}" in low or (M == 0 and has_any(low, num_words(H)))
            or (has_any(low, num_words(H)) and has_any(low, num_words(M)))
            or (M == 15 and "quarter past" in low and has_any(low, num_words(H)))
            or (M == 30 and "half past" in low and has_any(low, num_words(H)))
            or (M == 45 and "quarter to" in low and has_any(low, num_words(nxt)))
            or (M > 30 and f"{cardinal(60 - M)} to" in low and has_any(low, num_words(nxt))))


def fact_checks(src, out):
    """Numbers (incl. clock times), weekdays/months and mid-sentence names of the source, found in the output."""
    low = out.lower().replace("-", " ")
    numbers, dates, names = [], [], []
    rest = src
    for m in re.finditer(r"\b(\d{1,2}):(\d{2})\b", src):
        numbers.append(time_ok(m.group(1), m.group(2), low))
        rest = rest.replace(m.group(0), " ")
    for m in re.finditer(r"\$?\d[\d,]*(?:\.\d+)?", rest):
        tok = m.group(0).replace("$", "").replace(",", "")
        if "." in tok:
            whole, frac = tok.split(".")
            numbers.append(tok in low or (tok == "0.25" and "quarter" in low) or (tok == "0.5" and "half" in low)
                           or (has_any(low, num_words(int(whole))) and has_any(low, num_words(int(frac)))))
        else:
            numbers.append(tok in low or has_any(low, num_words(int(tok))))
    for d in WEEKDAYS + MONTHS:
        present = (re.search(r"(\d\w*\s+May\b|\bMay\s+\d|\b(in|by|until|of|before|after|this|next|last)\s+May\b)", src)
                   if d == "may" else re.search(rf"\b{d}\b", src, re.I))
        if present:
            dates.append(d in low)
    for m in re.finditer(r"(?<![.?!]\s)(?<!^)\b([A-Z][a-z]{2,})\b", src):
        if m.group(1).lower() not in WEEKDAYS + MONTHS:
            names.append(m.group(1).lower() in low)
    return {"numbers_ok": all(numbers) if numbers else None, "dates_ok": all(dates) if dates else None,
            "names_ok": all(names) if names else None}


NEG = re.compile(r"\b(not|no|never|nobody|nothing|none|neither|nor|without|cannot|nowhere)\b|n't", re.I)
CONTRACTION = re.compile(r"\b(it|he|she|that|what|there|here|who|let|where|how|this)'s\b|\b\w+'(re|ve|ll|m)\b|n't\b"
                         r"|\b(i|he|she|we|they|you|it|that|who)'d\b", re.I)


def faithfulness(src, out):
    """Facts kept (the dataset's checks), negation kept, and the register's hard style rules."""
    n_sent = len([x for x in re.split(r"(?<=[.?!])\s+", out.strip()) if x])
    return {**fact_checks(src, out),
            "negation_ok": bool(NEG.search(out)) if NEG.search(src) else None,
            "style_ok": not CONTRACTION.search(out) and "!" not in out and len(out.split()) <= 60 and n_sent <= 5}


FAITH = ["numbers_ok", "dates_ok", "names_ok", "negation_ok"]          # facts: the verdict's faithfulness
STYLE = ["style_ok"]                                                   # the register's hard rules: reported only
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()


def score(records, outputs):
    rows = [{"id": r["id"], "source": r["source"], "reference": r["target"], "output": o, "chrf": round(chrf(o, r["target"]), 2),
             **faithfulness(r["source"], o)} for r, o in zip(records, outputs)]
    s = {"n": len(rows), "chrf_reference_only": round(statistics.mean(x["chrf"] for x in rows), 1),
         "distinct_outputs": len({x["output"] for x in rows})}
    for k in FAITH + STYLE:
        v = [x[k] for x in rows if x[k] is not None]
        s[k] = round(sum(v) / len(v), 3) if v else None
    return rows, s

## Cell 4 — Train one configuration

- **Recipe:** micro-batches of 4, 2-step accumulation, linear decay over `MAX_EPOCHS`, the same data order per epoch for every configuration.
- **Early stopping:** on validation cross-entropy, patience 3; the best epoch's LoRA is kept in memory.

In [ ]:
# Cell 4 — Train one configuration

STATES = {}
SUMMARY_FILE = f"{WORK}/sweep_{MODEL_TAG}_n{len(TRAIN)}.json"
SUMMARY = json.load(open(SUMMARY_FILE)) if hub_get(SUMMARY_FILE) else {}


def train(cfg):
    name = cfg_name(cfg)
    lora = build(cfg)
    steps = MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM))
    opt = torch.optim.AdamW(lora, lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / steps))
    log = [{"epoch": 0, "validation_ce": val_ce(VAL)}]
    best, since, t0 = (log[0]["validation_ce"], 0, None), 0, time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        torch.manual_seed(SEED + epoch)
        order = random.Random(SEED + epoch).sample(TRAIN, len(TRAIN))
        batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
        model.train()
        run = []
        for n, b in enumerate(batches, 1):
            loss = ce_loss(b)
            (loss / ACCUM).backward()
            run.append(loss.item())
            if n % ACCUM == 0 or n == len(batches):
                opt.step()
                sched.step()
                opt.zero_grad()
        log.append({"epoch": epoch, "train_ce": round(statistics.mean(run), 4), "validation_ce": val_ce(VAL)})
        if log[-1]["validation_ce"] < best[0]:
            best, since = (log[-1]["validation_ce"], epoch, [p.detach().to("cpu", copy=True) for p in lora]), 0
        else:
            since += 1
        if since >= PATIENCE:
            break
    STATES[name] = (cfg, best[2])
    SUMMARY[name] = {"cfg": cfg, "best_validation_ce": best[0], "best_epoch": best[1], "epochs_run": log[-1]["epoch"],
                     "log": log, "seconds": round(time.perf_counter() - t0)}
    save_json(SUMMARY, SUMMARY_FILE)
    print(f"{name}: best validation CE {best[0]} at epoch {best[1]} (ran {log[-1]['epoch']}) | "
          f"{SUMMARY[name]['seconds']}s")
    return best[0]

## Cell 5 — Stage 1: the main grid (18 configurations)

In [ ]:
# Cell 5 — Stage 1

for cfg in GRID1:
    if cfg_name(cfg) in SUMMARY:
        print(f"{cfg_name(cfg)}: done earlier, best validation CE {SUMMARY[cfg_name(cfg)]['best_validation_ce']}")
        continue
    train(cfg)
ranked = sorted((v["best_validation_ce"], k) for k, v in SUMMARY.items() if v["cfg"] in GRID1)
print("\nstage 1 ranking (validation CE):")
for ce, k in ranked:
    print(f"  {ce:.4f}  {k}  (epoch {SUMMARY[k]['best_epoch']})")

## Cell 6 — Stage 2: regularisers on the best stage-1 setting (4 configurations)

In [ ]:
# Cell 6 — Stage 2

BEST1 = SUMMARY[ranked[0][1]]["cfg"]
GRID2 = [dict(BEST1, neftune=nf, wd=wd) for nf in (0.0, 5.0) for wd in (0.0, 0.05)]
for cfg in GRID2:
    if cfg_name(cfg) in SUMMARY:
        print(f"{cfg_name(cfg)}: done, best validation CE {SUMMARY[cfg_name(cfg)]['best_validation_ce']}")
        continue
    train(cfg)
print("\nstage 2:", {cfg_name(c): SUMMARY[cfg_name(c)]["best_validation_ce"] for c in GRID2})

## Cell 7 — Top 3 overall: validation translations and fact checks (for the blind judgement)

In [ ]:
# Cell 7 — Top 3: translate validation, score facts, save outputs

overall = sorted((v["best_validation_ce"], k) for k, v in SUMMARY.items())
TOP = [k for _, k in overall[:TOP_K]]
OUT_FILE = f"{WORK}/sweep_{MODEL_TAG}_n{len(TRAIN)}_top.json"
TOP_RESULTS = json.load(open(OUT_FILE)) if hub_get(OUT_FILE) else {}
for k in TOP:
    if k in TOP_RESULTS:
        continue
    if k not in STATES:
        train(SUMMARY[k]["cfg"])                      # retrain if this job did not train it (resumed run)
    cfg, state = STATES[k]
    lora = build(cfg)
    for p, s in zip(lora, state):
        p.data.copy_(s.to(p.device))
    rows, s = score(VAL, translate(VAL))
    TOP_RESULTS[k] = {"cfg": cfg, "best_validation_ce": SUMMARY[k]["best_validation_ce"], "scores": s, "rows": rows}
    save_json(TOP_RESULTS, OUT_FILE)
    print(k, s)
print("\ntop configurations by validation CE:")
for ce, k in overall[:8]:
    print(f"  {ce:.4f}  {k}")